In [2]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv('marketing_spend.csv')

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   month         180 non-null    str    
 1   year          180 non-null    int64  
 2   quarter       180 non-null    str    
 3   channel       180 non-null    str    
 4   spend         177 non-null    float64
 5   impressions   177 non-null    float64
 6   clicks        177 non-null    float64
 7   conversions   180 non-null    int64  
 8   cac           180 non-null    float64
 9   budget_share  180 non-null    float64
dtypes: float64(5), int64(2), str(3)
memory usage: 14.2 KB


In [8]:
df.head(30)

,month,year,quarter,channel,spend,impressions,clicks,conversions,cac,budget_share
0,2022-01,2022,Q1,Paid Social,10410.21,2884015.0,69837.0,247,42.00,52.0
1,2022-01,2022,Q1,Google Ads,6537.01,2137990.0,24324.0,172,38.00,28.0
2,2022-01,2022,Q1,Organic,736.47,238259.0,6538.0,92,8.00,5.0
3,2022-01,2022,Q1,Email,1166.59,371112.0,8389.0,194,6.00,7.0
4,2022-01,2022,Q1,Referral,1477.56,269102.0,4221.0,105,14.00,5.0
5,2022-02,2022,Q1,Paid Social,10954.62,2617201.0,70047.0,260,42.00,52.0
6,2022-02,2022,Q1,Google Ads,6027.57,1556452.0,28087.0,158,38.00,28.0
7,2022-02,2022,Q1,Organic,674.33,115637.0,1036.0,84,8.00,5.0
8,2022-02,2022,Q1,Email,1066.69,318854.0,5130.0,177,6.00,7.0
9,2022-02,2022,Q1,Referral,1483.15,456618.0,12486.0,105,14.00,5.0


In [10]:
# Check unique values
df['month'].unique()

# Check format consistency (should all be YYYY-MM)
df['month'].str.contains(r'^\d{4}-\d{2}$').all()

# Check for duplicates or unexpected patterns
df['month'].value_counts()

month
2022-01    5
2022-02    5
2022-03    5
2022-04    5
2022-05    5
2022-06    5
2022-07    5
2022-08    5
2022-09    5
2022-10    5
2022-11    5
2022-12    5
2023-01    5
2023-02    5
2023-03    5
2023-04    5
2023-05    5
2023-06    5
2023-07    5
2023-08    5
2023-09    5
2023-10    5
2023-11    5
2023-12    5
2024-01    5
2024-02    5
2024-03    5
2024-04    5
2024-05    5
2024-06    5
2024-07    5
2024-08    5
2024-09    5
2024-10    5
2024-11    5
2024-12    5
Name: count, dtype: int64

In [12]:
# Check unique years
df['year'].unique()

# Verify it matches the month column
(df['year'] == df['month'].str[:4].astype(int)).all()

# Check for any unexpected values
df['year'].value_counts()

# Check dtype
df['year'].dtype

dtype('int64')

In [14]:
# Check unique years and their counts
df['year'].value_counts().sort_index()

# Verify year matches the month column
(df['year'] == df['month'].str[:4].astype(int)).all()

np.True_

In [16]:
# Check unique quarters
df['quarter'].unique()

# Check value counts
df['quarter'].value_counts().sort_index()

# Verify quarter matches the month
df['month_quarter'] = df['month'].str[-2:].astype(int).apply(
    lambda x: f'Q{(x-1)//3 + 1}'
)
(df['quarter'] == df['month_quarter']).all()

np.True_

In [18]:
# Check unique channels
df['channel'].unique()

# Check value counts
df['channel'].value_counts()

# Check for whitespace or case issues
df['channel'].str.strip().str.lower().value_counts()

channel
paid social    36
google ads     36
organic        36
email          36
referral       36
Name: count, dtype: int64

In [20]:
# Check original unique values (before lowercasing)
df['channel'].unique()

# Check for leading/trailing whitespace
df['channel'].str.strip() == df['channel']  # Should all be True if no whitespace

0      True
1      True
2      True
3      True
4      True
       ... 
175    True
176    True
177    True
178    True
179    True
Name: channel, Length: 180, dtype: bool

In [24]:
df[df['spend'].isna()][['month', 'channel', 'spend']]

,month,channel,spend


In [25]:
# Calculate median spend by channel
df['spend_median_by_channel'] = df.groupby('channel')['spend'].transform('median')

# Fill missing spend with median of their respective channel
df.loc[df['spend'].isna(), 'spend'] = df.loc[df['spend'].isna(), 'spend_median_by_channel']

# Drop the temporary column
df.drop('spend_median_by_channel', axis=1, inplace=True)

# Verify no more missing
df['spend'].isna().sum()  # Should be 0

np.int64(0)

In [29]:
df[df['impressions'].isna()][['month', 'channel', 'impressions']]

,month,channel,impressions


In [30]:
df['impressions_median_by_channel'] = df.groupby('channel')['impressions'].transform('median')
df.loc[df['impressions'].isna(), 'impressions'] = df.loc[df['impressions'].isna(), 'impressions_median_by_channel']
df.drop('impressions_median_by_channel', axis=1, inplace=True)
df['impressions'].isna().sum()  # Should be 0

np.int64(0)

In [34]:
df[df['clicks'].isna()][['month', 'channel', 'clicks']]

,month,channel,clicks


In [35]:
df['clicks_median_by_channel'] = df.groupby('channel')['clicks'].transform('median')
df.loc[df['clicks'].isna(), 'clicks'] = df.loc[df['clicks'].isna(), 'clicks_median_by_channel']
df.drop('clicks_median_by_channel', axis=1, inplace=True)
df['clicks'].isna().sum()  # Should be 0

np.int64(0)

In [37]:
# Check for negative values
df['conversions'].min()

# Check summary statistics
df['conversions'].describe()

# Check for any zero values (might be valid)
df['conversions'].value_counts().head(10)

conversions
105    4
65     4
60     4
59     4
90     3
159    3
82     3
205    3
73     3
64     3
Name: count, dtype: int64

In [42]:
df['conversions'].min()
# Find rows with cac = 0
df[df['cac'] == 0][['month', 'channel', 'spend', 'conversions', 'cac']]

# Also check for negative values
df['cac'].min()

np.float64(0.0)

In [47]:
df[df['cac'] == 0][['month', 'channel', 'spend', 'conversions', 'cac']]

,month,channel,spend,conversions,cac


In [49]:
df.loc[df['cac'] == 0, 'cac'] = df.loc[df['cac'] == 0, 'spend'] / df.loc[df['cac'] == 0, 'conversions']

# Verify no more zeros
df['cac'].min()  # Should be > 0

np.float64(6.0)

In [51]:
# Check summary statistics
df['budget_share'].describe()

# Check minimum and maximum
df['budget_share'].min(), df['budget_share'].max()

# Check if all values are positive
df['budget_share'].min() > 0

np.True_

In [53]:
df['budget_share'].max()

np.float64(52.0)

In [55]:
df.head(30)

,month,year,quarter,channel,spend,impressions,clicks,conversions,cac,budget_share,month_quarter
0,2022-01,2022,Q1,Paid Social,10410.21,2884015.0,69837.0,247,42.000000,52.0,Q1
1,2022-01,2022,Q1,Google Ads,6537.01,2137990.0,24324.0,172,38.000000,28.0,Q1
2,2022-01,2022,Q1,Organic,736.47,238259.0,6538.0,92,8.000000,5.0,Q1
3,2022-01,2022,Q1,Email,1166.59,371112.0,8389.0,194,6.000000,7.0,Q1
4,2022-01,2022,Q1,Referral,1477.56,269102.0,4221.0,105,14.000000,5.0,Q1
5,2022-02,2022,Q1,Paid Social,10954.62,2617201.0,70047.0,260,42.000000,52.0,Q1
6,2022-02,2022,Q1,Google Ads,6027.57,1556452.0,28087.0,158,38.000000,28.0,Q1
7,2022-02,2022,Q1,Organic,674.33,115637.0,1036.0,84,8.000000,5.0,Q1
8,2022-02,2022,Q1,Email,1066.69,318854.0,5130.0,177,6.000000,7.0,Q1
9,2022-02,2022,Q1,Referral,1483.15,456618.0,12486.0,105,14.000000,5.0,Q1


In [57]:
# Drop the temporary month_quarter column first
# df.drop('month_quarter', axis=1, inplace=True)

# Export to CSV (without index)
df.to_csv('marketing_data_cleaned.csv', index=False)

In [58]:
# Read back and check
df_check = pd.read_csv('marketing_data_cleaned.csv')
df_check.info()

<class 'pandas.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   month          180 non-null    str    
 1   year           180 non-null    int64  
 2   quarter        180 non-null    str    
 3   channel        180 non-null    str    
 4   spend          180 non-null    float64
 5   impressions    180 non-null    float64
 6   clicks         180 non-null    float64
 7   conversions    180 non-null    int64  
 8   cac            180 non-null    float64
 9   budget_share   180 non-null    float64
 10  month_quarter  180 non-null    str    
dtypes: float64(5), int64(2), str(4)
memory usage: 15.6 KB
